# L6 - Datadog Quality Evaluations

## What You'll Learn

- How to score an agent's real response with a **code-based Bedrock LLM-as-judge** and submit the scores to Datadog
- How a deterministic ground-truth check complements the judge (and stays AI-Guard-aware)
- How to enable a **managed evaluation** (Language Mismatch) that Datadog runs with no judge model

## What Are Evaluations?

Evaluations measure the quality of your LLM application's outputs:

| Evaluation | What it checks | Example |
|----------------|---------------|----------|
| **Relevance** | Did the agent answer the actual question? | User asks about a refund, agent talks about shipping |
| **Faithfulness** | Is the response grounded, not fabricated? | Agent invents an order status the data doesn't support |
| **Verdict** | Did it answer, refuse, error, or get blocked? | A security block is NOT a quality failure |

## Prerequisites

- Notebook 3 completed (agents emitting Agent Observability spans)
- `ddtrace` + `boto3` installed; `DD_API_KEY` / `DD_SITE` in env or SSM
- **Bedrock model access** - Step 4's judge calls Claude directly (uses `harness_model_id` from SSM)


## License Details

Refer to LICENSE file in the main folder for license details.

## Step 1: Setup

In [ ]:
%pip install --quiet ddtrace boto3 --upgrade

## Step 2: Enable Agent Observability (the LLM Obs intake)

The specialist agents you deployed in L3 already stream LLM Obs spans through their
**trace-agent sidecar** — that production instrumentation is running and is left untouched
here. **This notebook does not re-instrument or redeploy the agents.**

This notebook runs in your **kernel**, which has no sidecar. To demonstrate the
*evaluation* workflow it enables a **separate** LLM Obs client in **agentless mode**
(`agentless_enabled=True`), which sends straight to Datadog using `DD_API_KEY`. It is used
only to (1) create one demonstration span around a single Harness call (Step 3) and
(2) attach evaluation scores to that span (Steps 4–5). So there are two independent LLM
Obs paths:

| Source | Instrumentation | Scope | Purpose |
|---|---|---|---|
| Deployed agents (L3) | trace-agent **sidecar** | continuous, production | live agent / LLM / tool spans |
| This notebook (L6 nb4) | **agentless** client, this kernel | one span, this session | demonstrate submitting evaluations |

The evaluations you submit below attach to the span **this notebook** creates — not to the
deployed agents' production spans. The managed eval in Step 7 then grades the spans this
notebook emits, server-side.

In [ ]:
import os, time, uuid
import boto3
from botocore.config import Config

# Load DD_API_KEY from SSM and set env var BEFORE importing ddtrace
REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
SSM_PREFIX = '/anycompany/agentcore'

DD_API_KEY = os.environ.get('DD_API_KEY', '')
DD_SITE = os.environ.get('DD_SITE', 'datadoghq.com')

if not DD_API_KEY:
    _ssm = boto3.client('ssm', region_name=REGION)
    try:
        DD_API_KEY = _ssm.get_parameter(Name=SSM_PREFIX+'/dd_api_key', WithDecryption=True)['Parameter']['Value']
        DD_SITE = _ssm.get_parameter(Name=SSM_PREFIX+'/dd_site')['Parameter']['Value']
        print('Loaded Datadog keys from SSM')
    except Exception as e:
        raise RuntimeError('Set DD_API_KEY env var or run l3-orchestration/1_setup_resources.ipynb Step 6 first. SSM error: ' + str(e))

# Set env vars BEFORE importing ddtrace (it checks at import time)
os.environ['DD_API_KEY'] = DD_API_KEY
os.environ['DD_SITE'] = DD_SITE

# Enable LLM Observability in AGENTLESS mode. This notebook has no trace-agent sidecar,
# so spans AND evaluations go straight to Datadog's LLM Obs intake using DD_API_KEY. This
# single channel carries both the span we create in Step 3 and the evaluations we submit in
# Steps 4-5; the managed eval in Step 7 then runs server-side on the spans that arrive here.
# (The deployed agents use the sidecar instead - this is notebook-side only.)
from ddtrace.llmobs import LLMObs

LLMObs.enable(
    ml_app='anycompany-customer-support',
    integrations_enabled=True,
    agentless_enabled=True,   # direct-to-Datadog intake; no sidecar in the notebook kernel
)
print('Agent Observability enabled')
print('  ml_app: anycompany-customer-support')
print('  site:   ' + DD_SITE)


## Step 3: Generate a Traced Interaction

Invoke the Harness and capture the span so we can attach evaluations to it.

In [ ]:
# Load AWS config from SSM
ssm = boto3.client('ssm', region_name=REGION)
cognito = boto3.client('cognito-idp', region_name=REGION)
agentcore = boto3.client('bedrock-agentcore', region_name=REGION, config=Config(read_timeout=300))

def get_ssm(n):
    return ssm.get_parameter(Name=n, WithDecryption=True)['Parameter']['Value']

cfg = {k: get_ssm(f'{SSM_PREFIX}/{k}') for k in ['harness_arn','cognito_client_id','user_password','model_id']}

# Authenticate
cognito.initiate_auth(ClientId=cfg['cognito_client_id'], AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={'USERNAME':'gold_customer','PASSWORD':cfg['user_password']})
print('Authenticated as gold_customer')

SESSION_ID = str(uuid.uuid4()).upper()
ACTOR_ID = 'CUST-789'
TEST_PROMPT = 'What is the status of order ORD-10001?'

print(f'Prompt: {TEST_PROMPT}')

with LLMObs.agent(name='anycompany-orchestrator', session_id=SESSION_ID) as agent_span:
    LLMObs.annotate(span=agent_span, input_data=TEST_PROMPT,
        metadata={'customer_id': ACTOR_ID, 'tier': 'gold'})

    with LLMObs.llm(name='bedrock-model-call', model_name=cfg.get('model_id','claude-sonnet'),
                    model_provider='aws_bedrock') as llm_span:
        t0 = time.time()
        resp = agentcore.invoke_harness(
            harnessArn=cfg['harness_arn'], runtimeSessionId=SESSION_ID, actorId=ACTOR_ID,
            messages=[{'role':'user','content':[{'text':f'[Authenticated customer: {ACTOR_ID} | Tier: gold]\n\n{TEST_PROMPT}'}]}])

        full_text = ''
        for ev in resp['stream']:
            if 'contentBlockDelta' in ev:
                full_text += ev['contentBlockDelta'].get('delta',{}).get('text','')

        LLMObs.annotate(span=llm_span,
            input_data=[{'role':'user','content':TEST_PROMPT}],
            output_data=[{'role':'assistant','content':full_text}],
            metrics={'latency_ms': round((time.time()-t0)*1000)})

    LLMObs.annotate(span=agent_span, output_data=full_text)
    # Export the span for evaluation (must be done inside the with block)
    traced_span = LLMObs.export_span(agent_span)

# Span exported (must be done while span context is available)
print(f'Response: {full_text[:300]}...')
print(f'\nSpan captured for evaluation.')


## Step 4: Score the Response with a Bedrock LLM-as-Judge

We ask a **Bedrock model to judge the agent's actual response** against the user's
question - returning a verdict (`answered` / `refused` / `blocked` / `error`) plus
relevance and faithfulness scores - and submit those scores to Datadog.

The rubric treats a security block as `blocked` rather than a quality failure, so an AI
Guard denial isn't mis-scored as a low-quality answer.


In [ ]:
# --- Score the REAL agent response with a Bedrock LLM-as-judge, then submit the scores ---
# How this reaches Datadog: WE compute the scores (the Bedrock call below), then
# LLMObs.submit_evaluation attaches each one to the exported span (trace_id + span_id from
# Step 3) via the agentless intake. Datadog STORES and DISPLAYS them on that span's
# Evaluations tab and lets you filter/alert on them - it does not compute these scores.
import json
bedrock = boto3.client('bedrock-runtime', region_name=REGION)
JUDGE_MODEL = get_ssm(f'{SSM_PREFIX}/harness_model_id')  # bare Bedrock model id (published by L2)

JUDGE_SYSTEM = (
    "You are a strict QA judge for a retail customer-support agent. "
    "Given the USER question and the AGENT response, reply with ONLY compact JSON:\n"
    '{"verdict":"answered|refused|blocked|error","relevance":<0-1 or null>,'
    '"faithfulness":<0-1 or null>,"reason":"<one sentence>"}\n'
    "Guidance:\n"
    "- blocked: the response was stopped by a security guard (mentions blocked / AI Guard / "
    "circuit breaker). Set relevance and faithfulness to null.\n"
    "- error: the response is a system/tool error, not an answer. Set scores to null.\n"
    "- refused: the agent declined a legitimate request.\n"
    "- answered: a genuine answer. relevance = did it address the question; "
    "faithfulness = internally consistent, not fabricating order data.\n"
    "Return JSON only, no prose."
)

def judge(question, answer):
    resp = bedrock.converse(
        modelId=JUDGE_MODEL,
        system=[{'text': JUDGE_SYSTEM}],
        messages=[{'role': 'user',
                   'content': [{'text': f'USER question:\n{question}\n\nAGENT response:\n{answer}'}]}],
        inferenceConfig={'maxTokens': 300, 'temperature': 0.0},
    )
    txt = resp['output']['message']['content'][0]['text']
    return json.loads(txt[txt.find('{'): txt.rfind('}') + 1])

try:
    verdict = judge(TEST_PROMPT, full_text)
    print('Judge verdict:', json.dumps(verdict, indent=2))

    # Submit REAL evaluations derived from the judge output.
    LLMObs.submit_evaluation(span=traced_span, label='answer_verdict',
                             metric_type='categorical', value=str(verdict.get('verdict', 'unknown')))
    print(f"Submitted: answer_verdict = {verdict.get('verdict')}")
    for label in ('relevance', 'faithfulness'):
        v = verdict.get(label)
        if isinstance(v, (int, float)):
            LLMObs.submit_evaluation(span=traced_span, label=label, metric_type='score', value=float(v))
            print(f'Submitted: {label} = {v}')
        else:
            print(f'Skipped {label} (verdict={verdict.get("verdict")}, no numeric score)')
except Exception as e:
    print(f'Judge failed: {e}')
    print('(Check Bedrock model access and that harness_model_id exists in SSM.)')


## Step 5: Deterministic Ground-Truth Check (and submit it)

The LLM judge generalizes; a **deterministic** check is cheap and exact when you *have*
ground truth (here we know `ORD-10001` should be `delivered`). We submit its result too,
so Datadog carries both signals.

> **AI-Guard-aware:** if AI Guard denied the tool output, the response won't contain the
> expected status. A naive substring match would then record a **faithfulness failure** -
> but that's a *security block*, not a quality problem. We detect the denial marker first
> and record it as `blocked` instead of scoring it 0.


In [ ]:
# Deterministic ground-truth check, AI-Guard-aware, submitted alongside the LLM judge.
# Same Datadog path as Step 4 (LLMObs.submit_evaluation -> attached to the same span); only
# the score SOURCE differs here - plain Python (ground-truth match), not an LLM.
AI_GUARD_DENIAL_MARKERS = ('blocked by ai guard', 'aiguardabort', 'denied by ai guard',
                           'ai guard security circuit breaker')

def response_blocked_by_ai_guard(response_text):
    t = (response_text or '').lower()
    return any(marker in t for marker in AI_GUARD_DENIAL_MARKERS)

def evaluate_order_response(response_text, order_id, expected_status):
    # Returns a 'blocked' verdict when AI Guard denied the output, so it is NOT recorded
    # as a faithfulness=0.0 failure (that would be a false negative).
    if response_blocked_by_ai_guard(response_text):
        return {'blocked_by_ai_guard': True, 'faithfulness': None, 'correctness': 'blocked'}
    if expected_status.lower() in response_text.lower():
        return {'blocked_by_ai_guard': False, 'faithfulness': 1.0, 'correctness': 'correct'}
    return {'blocked_by_ai_guard': False, 'faithfulness': 0.0, 'correctness': 'incorrect'}

# We know ORD-10001 should be 'delivered'.
scores = evaluate_order_response(full_text, 'ORD-10001', 'delivered')
print('Deterministic ground-truth check:')
if scores['blocked_by_ai_guard']:
    LLMObs.submit_evaluation(span=traced_span, label='ground_truth_match',
                             metric_type='categorical', value='blocked')
    print('  AI Guard DENIED -> recorded as "blocked" (NOT a faithfulness failure).')
else:
    LLMObs.submit_evaluation(span=traced_span, label='ground_truth_match',
                             metric_type='categorical', value=scores['correctness'])
    LLMObs.submit_evaluation(span=traced_span, label='faithfulness_exact',
                             metric_type='score', value=scores['faithfulness'])
    print(f"  ground_truth_match = {scores['correctness']}")
    print(f"  faithfulness_exact = {scores['faithfulness']}")

print()
print('This exact check complements the Step 4 LLM judge: deterministic where you have')
print('ground truth, while the judge generalizes to open-ended responses.')


## Step 6: Flush & View in Datadog

In [ ]:
LLMObs.flush()
time.sleep(5)

print(f'View evaluations at: https://app.{DD_SITE}/llm/traces')
print()
print('In the UI, click on any trace to see:')
print('  - Evaluation scores attached to each span')
print('  - Score trends over time')
print('  - Filter traces by evaluation score (e.g. faithfulness < 0.5)')


### Evaluations in Datadog UI

Click a trace in **AI Observability > Investigate > Trace > Evaluations** to see the evaluation scores
attached to each span.

![Datadog Evaluations](datadog_evaluations.png)


## Step 7: Managed Evaluations (Datadog-run, no code)

Datadog can also run evaluations **server-side** on every new span. Two flavors:

> **Key difference from Steps 4-5:** there *you* computed a score and submitted it; here
> **Datadog computes** the evaluation itself on the spans your app already sends.

- **Zero-model managed evals** - e.g. **Language Mismatch** (and Sensitive Data Scanning).
  Datadog runs them itself, with **no judge model to configure**.
- **LLM-as-judge templates** - Failure to Answer, Topic Relevancy, Sentiment, Toxicity, ...
  Each needs a **judge model connected in your Datadog org** (Bedrock via the AWS
  integration, or an OpenAI/Anthropic key).

### Step 7a: Create the evaluation in Datadog (do this first - UI only)

Managed evaluations are defined in the Datadog UI; there is **no API or code** to create
them, so this must be done by hand **before** running the cell below:

1. Go to **AI Observability > Evaluations > Create Evaluation**.
2. Choose **Language Mismatch** (flags responses whose language differs from the prompt).
3. Set **Account & Model** to *None* (Language Mismatch needs no judge model).
4. Scope it to `ml_app:anycompany-customer-support`, sampling **100%**.
5. **Publish**.

### Step 7b: Emit spans for it to grade

**Only after** the eval is published, run the next cell. Language Mismatch grades only
spans that arrive **after** it exists - so if you run the cell before publishing, publish
the eval and then **re-run the cell**.


In [ ]:
# Emit spans for the Language Mismatch eval to grade: one MISMATCH (Spanish in / English
# out) and one MATCH (English / English).
#
# PREREQUISITE: publish the Language Mismatch eval in the Datadog UI FIRST (Step 7a).
# Managed evals grade only spans that arrive AFTER they exist - if you run this before
# publishing, publish the eval and then re-run this cell. There is NO submit_evaluation
# here: we only emit spans through the agentless intake; Datadog grades them server-side.
print('Reminder: publish Language Mismatch in the Datadog UI (Step 7a) BEFORE this cell.\n')

with LLMObs.llm(name='lang-mismatch-demo', model_name='synthetic', model_provider='workshop') as span:
    LLMObs.annotate(span=span,
        input_data=[{'role': 'user', 'content': '\u00bfCu\u00e1l es el estado de mi pedido ORD-10001? Lo necesito hoy.'}],
        output_data=[{'role': 'assistant', 'content': 'Your order ORD-10001 has been delivered.'}])
print('Emitted MISMATCH span (Spanish in / English out) -> expect FAIL')

with LLMObs.llm(name='lang-match-demo', model_name='synthetic', model_provider='workshop') as span:
    LLMObs.annotate(span=span,
        input_data=[{'role': 'user', 'content': 'What is the status of order ORD-10002?'}],
        output_data=[{'role': 'assistant', 'content': 'Your order ORD-10002 is out for delivery.'}])
print('Emitted MATCH span (English in / English out) -> expect PASS')

LLMObs.flush()
print('\nFlushed. Check AI Observability > Evaluations > Language Mismatch in ~1-3 min:')
print('one span flagged (mismatch) and one clean (match).')


### Language Mismatch Results in Datadog

After ~1-3 minutes, **AI Observability > Evaluations > Language Mismatch** shows the two
spans just emitted: one flagged as a **mismatch** (Spanish in / English out) and one
**clean** (English / English) - graded server-side by Datadog with no judge model.

![Datadog Language Mismatch eval](datadog_language_mismatch.png)


## Summary

| What We Did | How |
|---|---|
| Generated a traced interaction | `LLMObs.agent` + `LLMObs.llm` wrapping `invoke_harness` |
| Scored it with an LLM judge | Bedrock (Claude) rubric -> real relevance / faithfulness + verdict |
| Added a deterministic check | Ground-truth match, AI-Guard-block aware |
| Enabled a managed eval | Language Mismatch (Datadog-run, no judge model) |

### Workshop Complete!

You now have full observability for the multi-agent system:

| Notebook | Coverage |
|----------|----------|
| 1 - Multi-Agent Tracing | Distributed APM traces across services |
| 2 - Metrics & Cost | Span-derived metrics, token cost, monitors, dashboard |
| 3 - Agent Observability | AI-specific Agent / LLM / Tool spans |
| 4 - Quality Evaluations | LLM-as-judge + deterministic + managed evaluations |


## Cleanup

Uncomment and run the cell below to clean up resources from this notebook:
- Disable LLM Observability for this session
- Note: Evaluations submitted to Datadog are retained per your plan's retention
  policy and cannot be deleted via API. They will expire automatically.


In [ ]:
# === CLEANUP: Uncomment all lines below and run this cell ===

# from ddtrace.llmobs import LLMObs

# # Disable LLM Observability — stops sending new traces and evaluations
# LLMObs.disable()
# print('LLM Observability disabled.')
# print('No further traces or evaluations will be sent from this session.')
# print()
# print('Note: Evaluations already in Datadog will expire based on retention settings.')
# print('View existing evaluations at: https://app.datadoghq.com/llm/traces')

# print('\nCleanup complete.')
